In [2]:
import json
import pandas as pd

In [3]:
# Load cleaned JSON
with open("../data/springfield_locations_cleaned.json", "r", encoding="utf-8") as f:
    locations = json.load(f)

In [4]:
df = pd.DataFrame(locations)

In [7]:
df.head()

,location_name,location_type,description
0,$50 Marmalade,,A shop located in the promenade at Springfield...
1,All Night Gym,Gym,"A gym in Springfield, it is where Homer attemp..."
2,Alkali Flats,Natural landmark,"The Alkali Flats, also known as the Springfiel..."
3,All Creatures Great and Cheap,Pet store inside the Springfield Mall,A pet store located inside the Springfield Mal...
4,All Trees $75,Retail of Christmas trees,A retail shop in Springfield specializes in se...


In [8]:
from sentence_transformers import SentenceTransformer

c:\wamp64\www\the-simpsons-ai-analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# A a small, fast local model
model = SentenceTransformer('all-MiniLM-L6-v2')

c:\wamp64\www\the-simpsons-ai-analysis\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\james\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [10]:
# Generate embeddings for each location description
descriptions = df['description'].tolist()
embeddings = model.encode(descriptions, convert_to_tensor=True)

In [11]:
import faiss
import numpy as np

In [12]:
# Convert embeddings to numpy float32
embeddings_np = embeddings.cpu().detach().numpy().astype('float32')

In [13]:
# Create FAISS index
dim = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings_np)

In [14]:
def ask_question(question, top_k=3):
    # Embed the question
    q_emb = model.encode([question]).astype('float32')
    
    # Search the index
    D, I = index.search(q_emb, top_k)
    
    # Return the top matches
    results = []
    for i in I[0]:
        results.append({
            "location_name": df.iloc[i]['location_name'],
            "location_type": df.iloc[i]['location_type'],
            "description": df.iloc[i]['description']
        })
    return results

In [ ]:
# Example
question = "?"
answers = ask_question(question)
for a in answers:
    print(f"{a['location_name']} ({a['location_type']}): {a['description']}\n")

Cheers (Bar): A bar from the sitcom "Cheers" appears in Springfield when Homer considers drinking there after being kicked out of Moe's Tavern. He encounters characters like Sam Malone and Carla Tortelli, but is put off by their arguments.

Coffee Shop (Springfield Heights) (Coffee shop): A coffee shop located in Springfield Heights, it is where Marge had coffee with Patty and Selma before learning that Homer was acting as a wingman.

Homer's Hunting Club (): A hunting club operated by Homer, it was established to allow him to serve drinks in his garage. The club features a bar where Marge serves as the barkeep, and it has hosted a performance by the band R.E.M. Although it has only gone hunting once, Homer attempted various strategies to catch a turkey, including using a plate as bait.

